# Synthèse vocale (TTS) en éwé — MMS-TTS (VITS)

**TTS** = *Text-To-Speech* : l'inverse de l'ASR. On part d'un **texte** éwé et on produit du
**son** (une voix). On utilise **MMS-TTS** de Meta, et plus précisément le modèle
`facebook/mms-tts-ewe`, **déjà entraîné pour l'éwé** : pas besoin de fine-tuning pour
obtenir une voix correcte. Ce notebook explique **comment l'utiliser** et **comment ça
marche**, étape par étape.

## Comment marche un système TTS moderne (VITS) ?

`mms-tts-ewe` repose sur l'architecture **VITS**, qui va du texte au son **en une seule fois** :

1. **Tokenizer** : découpe le texte en caractères/symboles → `input_ids` ;
2. **Encodeur de texte** : transforme ces symboles en représentations ;
3. **Prédicteur de durée** : décide combien de temps dure chaque son (le rythme) ;
4. **Flow + décodeur** : génère les caractéristiques acoustiques ;
5. **Vocodeur (HiFi-GAN)** : produit la **forme d'onde** finale (le son que l'on entend).

```mermaid
flowchart LR
    A[Texte eve] -->|tokenizer caracteres| B[input_ids]
    B --> C[Encodeur de texte]
    C --> D[Predicteur de duree]
    D --> E[Flow / decodeur]
    E --> F[Vocodeur HiFi-GAN]
    F --> G[Forme d onde 16 kHz]
```

Un atout de VITS : il introduit un peu de **hasard** (variables latentes), donc deux
générations de la même phrase ne sont pas parfaitement identiques — la voix paraît plus
naturelle.


## 0. Installation

`transformers` fournit `VitsModel`. `scipy` sert à écrire un fichier `.wav`.

In [ ]:
!pip install -q transformers torch scipy

## 1. Charger le modèle MMS-TTS éwé

`VitsModel` contient toute l'architecture (encodeur → vocodeur). Le `tokenizer` associé
sait découper le texte éwé en symboles connus du modèle.

In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer

MODEL_NAME = "facebook/mms-tts-ewe"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VitsModel.from_pretrained(MODEL_NAME).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SAMPLING_RATE = model.config.sampling_rate   # generalement 16000 Hz
print("Modele charge. Sampling rate :", SAMPLING_RATE, "Hz")
print("Romanisation requise (is_uroman) :", getattr(tokenizer, "is_uroman", False))


## 2. Comprendre la tokenisation TTS

Contrairement à un modèle de traduction (qui découpe en sous-mots), un modèle TTS découpe
souvent au niveau du **caractère/phonème** : chaque symbole correspond à un son.

> Si `is_uroman` valait `True`, il faudrait d'abord **romaniser** le texte (outil `uroman`).
> Pour l'éwé (écriture latine), c'est généralement inutile.

In [ ]:
texte = "Ŋdi nyuie, aleke nèfɔ ŋdi sia?"   # 'Bonjour, comment vas-tu ce matin ?'
inputs = tokenizer(texte, return_tensors="pt")

print("Texte      :", texte)
print("input_ids  :", inputs["input_ids"][0].tolist())
print("Nb symboles:", inputs["input_ids"].shape[1])


## 3. Générer la parole

Un simple passage avant (`model(**inputs)`) renvoie `waveform` : un tableau de nombres,
la **forme d'onde**. On l'écrit dans un `.wav` et on l'écoute directement dans le notebook.

In [ ]:
import scipy.io.wavfile
from IPython.display import Audio

inputs = tokenizer(texte, return_tensors="pt").to(device)
with torch.no_grad():
    waveform = model(**inputs).waveform

audio = waveform.squeeze().cpu().numpy()
scipy.io.wavfile.write("ewe_tts.wav", rate=SAMPLING_RATE, data=audio)
print("Fichier ecrit : ewe_tts.wav  (", round(len(audio) / SAMPLING_RATE, 2), "s )")

Audio(audio, rate=SAMPLING_RATE)


## 4. Contrôler la voix

Trois attributs du modèle ajustent le rendu :

- **`speaking_rate`** : vitesse d'élocution (plus grand = plus rapide) ;
- **`noise_scale`** : variabilité/expressivité de la voix ;
- **`noise_scale_duration`** : variabilité du rythme.

On les modifie puis on régénère pour comparer.

In [ ]:
model.speaking_rate = 1.2          # un peu plus rapide
model.noise_scale = 0.667          # expressivite par defaut

with torch.no_grad():
    waveform2 = model(**inputs).waveform
audio2 = waveform2.squeeze().cpu().numpy()
Audio(audio2, rate=SAMPLING_RATE)


## 5. Synthèse par lot

On génère plusieurs phrases d'affilée et on sauvegarde un fichier par phrase.

In [ ]:
phrases = [
    "Akpe na wo.",                       # 'Merci.'
    "Mawu ɖe gbe foo.",                  # extrait biblique
    "Ŋdi nyuie na mi katã.",             # 'Bonjour a tous.'
]

model.speaking_rate = 1.0
for i, p in enumerate(phrases):
    ins = tokenizer(p, return_tensors="pt").to(device)
    with torch.no_grad():
        w = model(**ins).waveform.squeeze().cpu().numpy()
    nom = f"ewe_tts_{i:02d}.wav"
    scipy.io.wavfile.write(nom, rate=SAMPLING_RATE, data=w)
    print(f"  {nom}  <-  {p}")


## 6. (Avancé) Fine-tuner un modèle TTS

Le modèle pré-entraîné suffit dans la plupart des cas. Si l'on voulait **adapter la voix**
(un locuteur précis, un style), il faudrait :

- un corpus **aligné** texte ↔ audio propre — notre dataset `ewe_asr` convient (mêmes
  paires texte/audio que pour l'ASR) ;
- entraîner VITS avec **deux pertes** : une perte acoustique (reconstruction) et une perte
  **adverse** (un discriminateur juge si le son paraît réel), plus une perte de durée ;
- beaucoup plus de calcul que pour LoRA/NLLB, car on apprend à **générer du signal**.

Outils recommandés (hors de ce notebook) : le script officiel **`finetune-hf-vits`** de
Hugging Face, ou **Coqui TTS**. La marche à suivre : rééchantillonner l'audio à la fréquence
du modèle, préparer un fichier `metadata` texte/audio, puis lancer l'entraînement VITS.


## 7. Conclusion — le pipeline vocal complet

Avec les trois briques de ce projet, on obtient une chaîne **voix → voix** :

```mermaid
flowchart LR
    A[Voix eve] -->|ASR Whisper| B[Texte eve]
    B -->|NLLB + LoRA| C[Texte fr / en]
    C -->|NLLB inverse| D[Texte eve]
    D -->|TTS MMS-VITS| E[Voix eve]
```

- **ASR** (`asr_ewe_whisper.ipynb`) : la voix éwé devient du texte ;
- **Traduction** (`traduction.ipynb`, `traduction_ewe_fra.ipynb`, `traduction_multilingue.ipynb`) :
  le texte éwé est traduit ;
- **TTS** (ce notebook) : un texte éwé est lu à voix haute.
